In [57]:
import pandas as pd
from prophet import Prophet
from DBD_1 import load_diesel
from DBD_1 import load_petrol
from DBD_1 import print_feature_dist
from DBD_1 import print_feature_vs_y_line_graphs
import numpy as np
from sklearn.metrics import root_mean_squared_error

## Load Data

In [54]:
diesel_df = load_diesel()
diesel_df["Total_Production_Lag1"] = diesel_df["Total_Production"].shift(1)
petrol_df = load_petrol()
petrol_df["Total_Production_Lag1"] = petrol_df["Total_Production"].shift(1)


/Users/hoyle/Documents/Uni/4th_Year/MOR441-AMX/fools-optimum/notebooks/DBD_1.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])
/Users/hoyle/Documents/Uni/4th_Year/MOR441-AMX/fools-optimum/notebooks/DBD_1.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


## Feature Selection

In [62]:
def evaluate_feature_set(df, feature_cols, target_col="Total_Fuel_Price",
                          window_size=12, n_windows=5, min_train=24):
    prophet_df = df[["Date", target_col] + feature_cols].copy()
    prophet_df = prophet_df.rename(columns={"Date": "ds", target_col: "y"})
    prophet_df = prophet_df.sort_values("ds").reset_index(drop=True)

    # Only the TARGET is shifted forward one month.
    # Features stay at row t — they already encode lag structure via naming
    # (e.g. BFP_Lag1 = last month's BFP), so no additional shift is needed
    # or correct here.
    prophet_df["y"] = prophet_df["y"].shift(-1)
    prophet_df = prophet_df.dropna().reset_index(drop=True)

    if len(prophet_df) < min_train + window_size:
        return np.nan

    rmses = []
    for i in range(n_windows):
        end = len(prophet_df) - i * window_size
        start = end - window_size
        if start - min_train < 0:
            break

        train_fold = prophet_df.iloc[:start]
        test_fold = prophet_df.iloc[start:end]
        if len(test_fold) == 0:
            continue

        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
            for col in feature_cols:
                m.add_regressor(col)
            m.fit(train_fold)

            fc = m.predict(test_fold[["ds"] + feature_cols])
            rmse = root_mean_squared_error(test_fold["y"], fc["yhat"])
            rmses.append(rmse)
        except Exception as e:
            print(f"  skipped a fold due to: {e}")
            continue

    return np.mean(rmses) if rmses else np.nan


def select_prophet_features(df, candidate_features, target_col="Total_Fuel_Price",
                             window_size=12, n_windows=5, min_train=24, max_features=10):
    remaining = list(candidate_features)
    selected = []

    best_rmse = evaluate_feature_set(df, selected, target_col, window_size, n_windows, min_train)

    improved = True
    while improved and remaining and len(selected) < max_features:
        improved = False
        scores = {
            feat: evaluate_feature_set(df, selected + [feat], target_col, window_size, n_windows, min_train)
            for feat in remaining
        }

        best_feat = min(scores, key=lambda k: scores[k] if not np.isnan(scores[k]) else np.inf)
        best_candidate_rmse = scores[best_feat]

        if not np.isnan(best_candidate_rmse) and best_candidate_rmse < best_rmse:
            selected.append(best_feat)
            remaining.remove(best_feat)
            best_rmse = best_candidate_rmse
            improved = True

    return selected, best_rmse

In [64]:
master_diesel_df = load_diesel()
master_diesel_df["Total_Production_Lag1"] = diesel_df["Total_Production"].shift(1)

exclude_cols = ["Date", "Total_Fuel_Price", "INDPRO", "GSCPI", "GECON", "Total_Production"]
candidate_features = [c for c in master_diesel_df.columns if c not in exclude_cols]

selected_diesel_features, final_rmse = select_prophet_features(master_diesel_df, candidate_features)

print(selected_diesel_features)
print(f"Final mean RMSE: {final_rmse:.4f}")

/Users/hoyle/Documents/Uni/4th_Year/MOR441-AMX/fools-optimum/notebooks/DBD_1.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])
13:48:30 - cmdstanpy - INFO - Chain [1] start processing
13:48:30 - cmdstanpy - INFO - Chain [1] done processing
13:48:30 - cmdstanpy - INFO - Chain [1] start processing
13:48:30 - cmdstanpy - INFO - Chain [1] done processing
13:48:31 - cmdstanpy - INFO - Chain [1] start processing
13:48:31 - cmdstanpy - INFO - Chain [1] done processing
13:48:31 - cmdstanpy - INFO - Chain [1] start processing
13:48:31 - cmdstanpy - INFO - Chain [1] done processing
13:48:31 - cmdstanpy - INFO - Chain [1] start processing
13:48:31 - cmdstanpy - INFO - Chain [1] done processing
13:48:31 - cmdstanpy - INFO - Chain [1] start processing
13:48:31 - cmdstanpy - INFO - Chain [1] done processing
13:48:31 - c

['BFP', 'Brent_LastWeekMean', 'Brent_MonthMean_Lag1', 'BFP_Delta_Lag1', 'Brent_LastWeekStd', 'BFP_Lag3', 'BFP_Delta_Lag6', 'USDZAR_LastWeekMean', 'USDZAR_Mean_Lag1', 'INDPRO_Lag3']
Final mean RMSE: 80.8275


In [65]:
prophet_diesel_df = petrol_df[["Date", "Total_Fuel_Price"]].rename(
    columns={"Date": "ds", "Total_Fuel_Price": "y"}
)
prophet_diesel_df = prophet_diesel_df.sort_values("ds").reset_index(drop=True)
prophet_diesel_df["y"] = prophet_diesel_df["y"].shift(-1)
prophet_diesel_df = prophet_diesel_df.dropna(subset=["y"]).reset_index(drop=True)

prophet_diesel_df.tail()

,ds,y
181,2026-02-01,2030.3
182,2026-03-01,2336.3
183,2026-04-01,2663.2
184,2026-05-01,2806.2
185,2026-06-01,2610.2


In [66]:
regressors = ['BFP', 'Brent_LastWeekMean', 'Brent_MonthMean_Lag1', 'BFP_Delta_Lag1', 'Brent_LastWeekStd', 'BFP_Lag3', 'BFP_Delta_Lag6', 'USDZAR_LastWeekMean', 'USDZAR_Mean_Lag1', 'INDPRO_Lag3']


for col in regressors:
    prophet_diesel_df[col] = diesel_df[col].shift(-1).values[:len(prophet_diesel_df)]  # align to same shift as y

prophet_diesel_df = prophet_diesel_df.dropna().reset_index(drop=True)

In [67]:
print_feature_dist(prophet_diesel_df)
print_feature_vs_y_line_graphs(prophet_diesel_df)

In [68]:
prophet_diesel_df

,ds,y,BFP,Brent_LastWeekMean,Brent_MonthMean_Lag1,BFP_Delta_Lag1,Brent_LastWeekStd,BFP_Lag3,BFP_Delta_Lag6,USDZAR_LastWeekMean,USDZAR_Mean_Lag1,INDPRO_Lag3
0,2011-07-01,1009.4,598.03,113.088333,114.678966,-12.00,2.394764,632.03,32.0,7.143857,6.802743,93.8880
1,2011-08-01,1018.4,598.03,108.120000,112.088276,15.00,1.465549,595.03,63.0,8.041786,6.984373,94.1720
2,2011-09-01,1054.4,635.03,110.957143,112.745357,0.00,1.416601,583.03,44.0,7.921757,7.437997,94.6360
3,2011-10-01,1077.4,671.03,108.755000,109.534667,37.00,2.214649,598.03,16.0,8.421929,7.958540,95.2749
4,2011-11-01,1066.4,707.03,108.156000,110.752759,36.00,0.991075,598.03,-37.0,8.178429,8.101867,95.1961
...,...,...,...,...,...,...,...,...,...,...,...,...
175,2026-02-01,2030.3,991.03,114.968571,70.160000,-56.00,8.482695,1133.03,-57.0,17.059043,16.031387,101.4941
176,2026-03-01,2336.3,2017.03,116.922857,94.781333,64.00,5.273420,983.03,-8.0,16.561714,16.541643,101.0388
177,2026-04-01,2663.2,2606.10,100.158333,116.748214,1026.00,5.808530,927.03,-19.0,16.379743,16.693410,101.9263
178,2026-05-01,2806.2,2021.03,72.888571,109.416071,589.07,2.490324,991.03,77.0,16.488629,16.490773,101.6172


Training Size Metrics

In [69]:
from sklearn.metrics import root_mean_squared_error

results = []
window_size = 12
n_windows = 5  # e.g. 5 separate 12-month test periods, walking backward

for i in range(n_windows):
    end = len(prophet_diesel_df) - i * window_size
    start = end - window_size
    if start - 24 < 0:  # ensure enough training history remains
        break

    train_fold = prophet_diesel_df.iloc[:start]
    test_fold = prophet_diesel_df.iloc[start:end]

    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    for col in regressors:
        m.add_regressor(col)
    m.fit(train_fold)

    fc = m.predict(test_fold[["ds"] + regressors])
    rmse = root_mean_squared_error(test_fold["y"], fc["yhat"])
    results.append({"window_end": test_fold["ds"].max(), "rmse": rmse})

pd.DataFrame(results)

13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:05 - cmdstanpy - INFO - Chain [1] done processing
13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:05 - cmdstanpy - INFO - Chain [1] done processing
13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:05 - cmdstanpy - INFO - Chain [1] done processing
13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:05 - cmdstanpy - INFO - Chain [1] done processing
13:53:05 - cmdstanpy - INFO - Chain [1] start processing
13:53:05 - cmdstanpy - INFO - Chain [1] done processing


,window_end,rmse
0,2026-06-01,266.979429
1,2025-06-01,179.342367
2,2024-06-01,62.193209
3,2023-06-01,95.571808
4,2022-06-01,125.216142
